# Backtest Performance & Risk Evaluation
This notebook evaluates 5 years of walk-forward backtest results (1,508 trading days, monthly rebalancing, 10 bps transaction costs) across Equal Weight, Minimum Variance, Risk Parity, and the S&P 500 benchmark.

In [ ]:
import sys
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

current_path = pathlib.Path('.').resolve()
project_root = current_path if (current_path / 'src').exists() else current_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

sns.set_theme(style='whitegrid', palette='husl')

export_dir = project_root / 'data' / 'exports'
perf_path = export_dir / 'strategy_performance.csv'
metrics_path = export_dir / 'risk_metrics.csv'
weights_path = export_dir / 'portfolio_weights.csv'

perf_raw = pd.read_csv(perf_path)
perf_df = perf_raw.pivot(index='Date', columns='Strategy', values='Normalized Value')
perf_df.index = pd.to_datetime(perf_df.index)

metrics_df = pd.read_csv(metrics_path, index_col=0)
weights_df = pd.read_csv(weights_path)
print("Backtest data loaded successfully.")

## 1. Strategy Equity Curves vs S&P 500 Benchmark

In [ ]:
plt.figure(figsize=(14, 8))
colors = {'Equal Weight': '#2563eb', 'Risk Parity': '#10b981', 'Minimum Variance': '#f59e0b', 'S&P 500': '#64748b'}
linestyles = {'Equal Weight': '-', 'Risk Parity': '-', 'Minimum Variance': '-', 'S&P 500': '--'}

for strat in perf_df.columns:
    plt.plot(perf_df.index, perf_df[strat], label=strat, 
             color=colors.get(strat, 'black'), 
             linestyle=linestyles.get(strat, '-'), 
             linewidth=2.5 if strat != 'S&P 500' else 2)

plt.title("Portfolio Equity Growth ($1.00 Invested on 2019-01-02)", fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.ylabel("Portfolio Value (Normalized)", fontsize=11)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 2. Strategy Risk & Return Comparison Table

In [ ]:
print("=== Comprehensive Risk & Performance Metrics ===")
print(metrics_df.to_string())

## 3. Drawdown Underwater Curves

In [ ]:
drawdowns = (perf_df / perf_df.cummax()) - 1

plt.figure(figsize=(14, 6))
for strat in drawdowns.columns:
    plt.plot(drawdowns.index, drawdowns[strat] * 100, label=strat, 
             color=colors.get(strat, 'black'), 
             linestyle=linestyles.get(strat, '-'),
             alpha=0.85)

plt.title("Historical Drawdown Series (Peak-to-Trough Decline %)", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 4. Rolling 21-Day Volatility

In [ ]:
daily_rets = perf_df.pct_change().dropna()
rolling_vol = daily_rets.rolling(window=21).std() * np.sqrt(252) * 100

plt.figure(figsize=(14, 6))
for strat in rolling_vol.columns:
    plt.plot(rolling_vol.index, rolling_vol[strat], label=strat, color=colors.get(strat, 'black'), alpha=0.8)

plt.title("21-Day Rolling Annualized Volatility (%)", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Annualized Volatility (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Risk-Return Scatter (Sharpe & Beta Positioning)

In [ ]:
plt.figure(figsize=(9, 6))
for strat, row in metrics_df.iterrows():
    plt.scatter(row['Ann. Volatility'] * 100, row['CAGR'] * 100, s=200, label=strat, color=colors.get(strat, 'black'))
    plt.annotate(strat, (row['Ann. Volatility'] * 100 + 0.2, row['CAGR'] * 100), fontsize=10, fontweight='bold')

plt.title("Risk vs Return Profile (2019-2024)", fontsize=14, fontweight='bold')
plt.xlabel("Annualized Volatility (%)")
plt.ylabel("CAGR (%)")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Portfolio Weights Over Time (Stacked Area Chart)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for i, strat in enumerate(['Equal Weight', 'Minimum Variance', 'Risk Parity']):
    df_s = weights_df[weights_df['Strategy'] == strat].pivot(index='Date', columns='Ticker', values='Weight')
    df_s.index = pd.to_datetime(df_s.index)
    axes[i].stackplot(df_s.index, df_s.T, labels=df_s.columns, alpha=0.85)
    axes[i].set_title(f"{strat} Weight Evolution (Monthly Rebalancing)", fontsize=12, fontweight='bold')
    axes[i].set_ylabel("Weight Fraction")
    if i == 0:
        axes[i].legend(loc='upper left', bbox_to_anchor=(1.01, 1), ncol=1, title="Ticker")

plt.xlabel("Rebalance Date")
plt.tight_layout()
plt.show()